<a href="https://colab.research.google.com/github/RoseWang-web/Data_Portfolio/blob/gh-pages/starter_housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import polars as pl
import requests

# ==============================================================================
# 1. LOAD MAIN HOUSING & RUN INTERNAL FEATURES
#==============================================================================
data_url = "https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing.csv"
print("🔄 Loading main housing dataset...")
df_housing = pl.read_csv(data_url, infer_schema_length=None)

# Standardize housing zip code column type to String right away
df_housing = df_housing.with_columns(pl.col("zipcode").cast(pl.String))

# Apply internal feature metrics
CURRENT_YEAR = 2026
df_housing = df_housing.with_columns([
    (CURRENT_YEAR - pl.col("yr_built")).alias("house_age"),
    pl.when(pl.col("yr_renovated") > 0).then(1).otherwise(0).cast(pl.Int8).alias("is_renovated"),
    (pl.col("sqft_living") / pl.col("sqft_lot")).alias("living_to_lot_ratio"),
    pl.when(pl.col("yr_renovated") > 0)
      .then(pl.col("yr_renovated") - pl.col("yr_built"))
      .otherwise(0)
      .alias("age_at_renovation")
])
df_housing
# ==============================================================================
# 2. PROCESS NEW LOCAL KING COUNTY GEOGRAPHIC DATA
# ==============================================================================
print("📂 Processing King County geographic CSV files...")

# File A: Process Transit Stop Point Data
transit_raw = pl.read_csv("C:\\Users\\Rose\\Downloads\\TRANSITSTOP_POINT_2648_6217742506826507293.csv", infer_schema_length=None)

transit_features = (
    transit_raw.filter(pl.col("ZIPCODE").is_not_null())
    .group_by("ZIPCODE")
    .len(name="transit_stop_count")
    .with_columns(
        pl.col("ZIPCODE").cast(pl.Int64).cast(pl.String).alias("zipcode")
    )
    .select(["zipcode", "transit_stop_count"])
)
transit_features

# File B: Process Master Zip Code Area Labels. Know what the zipcode city is.
zip_area_raw = pl.read_csv("C:\\Users\\Rose\\Downloads\\ZIPCODE_AREA_113_-7575923771333279216.csv", infer_schema_length=None)

city_features = (
    zip_area_raw.select(["ZIPCODE", "PREFERRED_CITY"])
    .with_columns(
        pl.col("ZIPCODE").cast(pl.Int64).cast(pl.String).alias("zipcode")
    )
    .select(["zipcode", "PREFERRED_CITY"])
    .unique(subset=["zipcode"])
)

# ==============================================================================
# 3. FETCH U.S. CENSUS BUREAU ACS DATA VIA API WITH SAFE FALLBACK
# ==============================================================================
print("🌐 Connecting to U.S. Census Bureau API...")
YEAR = "2022"
BASE_URL = f"https://api.census.gov/data/{YEAR}/acs/acs5"

variables = {
    "B19013_001E": "median_household_income",
    "B25077_001E": "median_home_value",
    "B25064_001E": "median_gross_rent",
    "B01003_001E": "total_population",
}

var_string = ",".join(variables.keys())
# We choose 53 because it is the official Federal Information Processing System (FIPS) numeric code for the State of Washington.
payload = {
    "get": var_string,
    "for": "zip code tabulation area:*",
    "in": "state:53"
}

# Define column names for structural alignment
census_columns = ["zipcode", "median_household_income", "median_home_value", "median_gross_rent", "total_population"]

try:
    response = requests.get(BASE_URL, params=payload, timeout=10)

    # Check if the content-type returned is actually JSON, not HTML error pages
    if response.status_code == 200 and "application/json" in response.headers.get("Content-Type", ""):
        raw_data = response.json()
        headers = raw_data[0]
        rows = raw_data[1:]
        census_raw_df = pl.DataFrame(rows, schema=headers)

        rename_mapping = {k: v for k, v in variables.items()}
        rename_mapping["zip code tabulation area"] = "zipcode"
        census_df = census_raw_df.rename(rename_mapping)
        if "state" in census_df.columns:
            census_df = census_df.drop("state")

        census_df = census_df.with_columns([
            pl.col("zipcode").cast(pl.String),
            pl.when(pl.col("median_household_income").cast(pl.Float64) >= 0).then(pl.col("median_household_income").cast(pl.Float64)).otherwise(None).alias("median_household_income"),
            pl.when(pl.col("median_home_value").cast(pl.Float64) >= 0).then(pl.col("median_home_value").cast(pl.Float64)).otherwise(None).alias("median_home_value"),
            pl.when(pl.col("median_gross_rent").cast(pl.Float64) >= 0).then(pl.col("median_gross_rent").cast(pl.Float64)).otherwise(None).alias("median_gross_rent"),
            pl.when(pl.col("total_population").cast(pl.Int64) >= 0).then(pl.col("total_population").cast(pl.Int64)).otherwise(None).alias("total_population"),
        ])
        print(f"✅ Live Census API Fetch Successful! Shape: {census_df.shape}")
    else:
        print("⚠️ Census Server returned an error page or key cap threshold was hit. Initializing automated fallback layout...")
        # Create an empty template that matches the correct schema structure
        census_df = pl.DataFrame(schema={col: (pl.String if col == "zipcode" else pl.Float64) for col in census_columns})
except Exception as e:
    print(f"⚠️ Network exception encountered. Initializing automated fallback layout...")
    census_df = pl.DataFrame(schema={col: (pl.String if col == "zipcode" else pl.Float64) for col in census_columns})

# ==============================================================================
# 4. CONSOLIDATE LOCAL GEOGRAPHIC AND CENSUS DATA
# ==============================================================================
print("🔗 Linking Housing Data with Local GIS Data and US Census Data...")
expanded_df = (
    df_housing
    .join(transit_features, on="zipcode", how="left")
    .join(city_features, on="zipcode", how="left")
    .join(census_df, on="zipcode", how="left")
)

# Impute and clean missing values (Baseline regional mappings if API fallback is active)
expanded_df = expanded_df.with_columns([
    pl.col("transit_stop_count").fill_null(0),
    pl.col("PREFERRED_CITY").fill_null("Unknown"),
    pl.col("median_household_income").fill_null(85000.0), # King County Median Income Default
    pl.col("median_home_value").fill_null(650000.0),      # King County Median Valuation Default
    pl.col("median_gross_rent").fill_null(18000.0),       # Regional Rental Baseline Default
    pl.col("total_population").fill_null(35000.0)         # Neighborhood Density Baseline Default
])
expanded_df
print("✅ Base pipeline matrix 'expanded_df' constructed and loaded into memory!")



🔄 Loading main housing dataset...
📂 Processing King County geographic CSV files...


FileNotFoundError: No such file or directory (os error 2): C:\Users\Rose\Downloads\TRANSITSTOP_POINT_2648_6217742506826507293.csv

🔄 Loading main housing dataset...
📂 Processing King County geographic CSV files...


FileNotFoundError: No such file or directory (os error 2): C:\Users\Rose\Downloads\TRANSITSTOP_POINT_2648_6217742506826507293.csv

NameError: ❌ 'expanded_df' is missing. Please run Cell 1 before running this time-series alignment step!